# Project: Time Series Analysis with Pandas

Analyze time series data using pandas datetime operations, resampling, rolling windows, and seasonal decomposition.

In [ ]:
import sys, os, subprocess
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    REPO_PATH = '/content/drive/MyDrive/data-science-mastery'
    if os.path.isdir(REPO_PATH):
        os.chdir(REPO_PATH)
        print(f'Working directory: {os.getcwd()}')
        if not os.path.isdir('Data') or len(os.listdir('Data')) < 5:
            subprocess.check_call([sys.executable, 'scripts/prepare_datasets.py'])
    else:
        REPO_PATH = '/content/data-science-mastery'
        if os.path.isdir(REPO_PATH):
            os.chdir(REPO_PATH)
        else:
            print('Repo not found. Run opencolab_setup.ipynb first.')

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline
print('Pandas version:', pd.__version__)

## Generate Time Series Data

In [ ]:
np.random.seed(42)
dates = pd.date_range('2022-01-01', '2024-12-31', freq='D')
n = len(dates)
trend = np.linspace(100, 200, n)
seasonality = 30 * np.sin(2 * np.pi * np.arange(n) / 365)
weekly = 15 * np.sin(2 * np.pi * np.arange(n) / 7)
noise = np.random.normal(0, 20, n)
sales = np.maximum(trend + seasonality + weekly + noise, 0)
df = pd.DataFrame({'date': dates, 'sales': sales}).set_index('date')
print('Date range:', df.index.min(), 'to', df.index.max())
print('Shape:', df.shape)
df.head()

## 1. Resampling and Aggregation

In [ ]:
resampled = {
    'Weekly': df.resample('W').mean(),
    'Monthly': df.resample('ME').mean(),
    'Quarterly': df.resample('QE').mean(),
}
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
df.plot(ax=axes[0,0], title='Daily', alpha=0.5, legend=False)
resampled['Weekly'].plot(ax=axes[0,1], title='Weekly', color='green', legend=False)
resampled['Monthly'].plot(ax=axes[1,0], title='Monthly', color='red', legend=False)
resampled['Quarterly'].plot(ax=axes[1,1], title='Quarterly', marker='o', legend=False)
plt.tight_layout()
plt.show()

## 2. Rolling Window Statistics

In [ ]:
df['roll_mean_7'] = df['sales'].rolling(7).mean()
df['roll_mean_30'] = df['sales'].rolling(30).mean()
df[['sales','roll_mean_7','roll_mean_30']].iloc[:365].plot(figsize=(14, 5))
plt.title('Sales with 7-day and 30-day Rolling Averages')
plt.grid(True, alpha=0.3)
plt.show()

## 3. Time-Based Indexing

In [ ]:
print(f'Sales in 2023: {df.loc["2023"].shape[0]} days')
print(f'Q1 2023 avg: {df.loc["2023-01":"2023-03", "sales"].mean():.1f}')
print(f'Jan 2024 avg: {df.loc["2024-01", "sales"].mean():.1f}')
print(f'2022 avg: {df.loc["2022", "sales"].mean():.1f}')
print(f'2023 avg: {df.loc["2023", "sales"].mean():.1f}')

## 4. Seasonal Decomposition

In [ ]:
from statsmodels.tsa.seasonal import seasonal_decompose
result = seasonal_decompose(df.loc['2023', 'sales'], model='additive', period=365)
fig = result.plot()
fig.set_size_inches(14, 10)
plt.tight_layout()
plt.show()

## 5. Lag Features and Date Features

In [ ]:
for lag in [1, 2, 3, 7, 14, 30]:
    df[f'lag_{lag}'] = df['sales'].shift(lag)
df['day_of_week'] = df.index.dayofweek
df['month'] = df.index.month
df['quarter'] = df.index.quarter
df['weekend'] = df['day_of_week'].isin([5,6]).astype(int)

weekly_pattern = df.groupby('day_of_week')['sales'].mean()
weekly_pattern.plot(kind='bar', figsize=(10, 4), color='steelblue')
plt.xlabel('Day of Week (0=Mon)')
plt.ylabel('Average Sales')
plt.title('Sales by Day of Week')
plt.grid(True, alpha=0.3)
plt.show()
print(weekly_pattern.round(1))

## Summary

- Resampling to different time frequencies
- Rolling window calculations
- Time-based slicing
- Seasonal decomposition
- Lag features and date features